# 04 - Strategy 1: random 75/25 split (Table 1)

For each field group (ASP, BAU, ASP+BAU), climate class (all years, dry, normal, wet) and feature case
(1-8), an XGBoost model with default hyperparameters is:

1. cross-validated with 5 folds on a random 75% of the pixels,
2. refitted on that 75% and scored on the remaining 25%.

The test scores are Table 1 of the paper and the cross-validation scores are Supplementary Table S6.
Yield is modelled in kg/ha.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

sys.path.append("../src")
from yieldml import CASES, TARGET, Y_MULT, YEAR_CLASS, by_year_class, load_all, scores
from sklearn.model_selection import KFold, train_test_split

In [2]:
TABLE_DIR = Path("../results/tables")
data = load_all()
for g, d in data.items():
    print(f"{g}: {len(d)} pixels, fields {sorted(d['FieldName'].unique())}")

ASP: 19712 pixels, fields ['S2', 'S4', 'S5', 'S6', 'SB1', 'SB4', 'SB5', 'SB7', 'SCD2', 'SCD3', 'SCD5', 'SCD6']
BAU: 18985 pixels, fields ['S3', 'S7', 'SB3', 'SB6', 'SCD4', 'SCD7']
ASP+BAU: 38697 pixels, fields ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'SB1', 'SB3', 'SB4', 'SB5', 'SB6', 'SB7', 'SCD2', 'SCD3', 'SCD4', 'SCD5', 'SCD6', 'SCD7']


## Train, cross-validate and test

In [3]:
CLASSES = ["All", "Dry", "Wet", "Normal"]
rows = []
for group, df in data.items():
    for cls in CLASSES:
        d = by_year_class(df, cls)
        for case, feats in CASES.items():
            X, y = d[feats].values, d[TARGET].values * Y_MULT
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

            cv = []
            for tr, va in KFold(n_splits=5, shuffle=True, random_state=42).split(X_train):
                m = XGBRegressor(objective="reg:squarederror", random_state=42).fit(X_train[tr], y_train[tr])
                cv.append(scores(y_train[va], m.predict(X_train[va])))
            cv = pd.DataFrame(cv).mean()

            m = XGBRegressor(objective="reg:squarederror", random_state=42).fit(X_train, y_train)
            test = scores(y_test, m.predict(X_test))
            rows.append({"Group": group, "Class": cls, "Case": case, "n": len(d),
                         **{f"{k}_CV": v for k, v in cv.items()},
                         **{f"{k}_Test": v for k, v in test.items()}})
        print(f"{group:8s} {cls:6s} done")

results = pd.DataFrame(rows)
results.to_csv(TABLE_DIR / "strategy1_all_scores.csv", index=False)

ASP      All    done
ASP      Dry    done
ASP      Wet    done
ASP      Normal done
BAU      All    done
BAU      Dry    done
BAU      Wet    done
BAU      Normal done
ASP+BAU  All    done
ASP+BAU  Dry    done
ASP+BAU  Wet    done
ASP+BAU  Normal done


## Table 1: test scores

In [4]:
table1 = results.pivot_table(index=["Group", "Case"], columns="Class",
                             values=["R2_Test", "RMSE_Test", "RRMSE_Test"])
table1 = table1.swaplevel(0, 1, axis=1)[[(c, m) for c in CLASSES
                                         for m in ["R2_Test", "RMSE_Test", "RRMSE_Test"]]]
table1 = table1.reindex(["ASP+BAU", "ASP", "BAU"], level=0)
table1.to_csv(TABLE_DIR / "table1_strategy1_test.csv")
table1.round({(c, "R2_Test"): 2 for c in CLASSES} | {(c, "RMSE_Test"): 0 for c in CLASSES}
             | {(c, "RRMSE_Test"): 1 for c in CLASSES})

Class             All                          Dry                       \
              R2_Test RMSE_Test RRMSE_Test R2_Test RMSE_Test RRMSE_Test   
Group   Case                                                              
ASP+BAU Case1    0.91     506.0       14.3    0.86     347.0       19.9   
        Case2    0.90     533.0       15.0    0.86     347.0       19.8   
        Case3    0.92     498.0       14.0    0.87     339.0       19.4   
        Case4    0.92     494.0       13.9    0.87     335.0       19.2   
        Case5    0.91     504.0       14.2    0.86     345.0       19.7   
        Case6    0.92     498.0       14.0    0.87     338.0       19.3   
        Case7    0.92     496.0       14.0    0.87     336.0       19.2   
        Case8    0.90     543.0       15.3    0.84     375.0       21.4   
ASP     Case1    0.93     505.0       14.1    0.88     308.0       20.2   
        Case2    0.92     540.0       15.1    0.87     316.0       20.7   
        Case3    0.93     503.0       14.1    0.87     320.0       21.0   
        Case4    0.93     501.0       14.0    0.87     317.0       20.8   
        Case5    0.93     501.0       14.0    0.87     317.0       20.8   
        Case6    0.93     495.0       13.8    0.86     323.0       21.2   
        Case7    0.93     496.0       13.8    0.87     312.0       20.5   
        Case8    0.92     534.0       14.9    0.83     363.0       23.8   
BAU     Case1    0.88     513.0       14.5    0.81     388.0       19.2   
        Case2    0.89     510.0       14.4    0.82     384.0       19.0   
        Case3    0.89     498.0       14.0    0.84     359.0       17.8   
        Case4    0.89     501.0       14.1    0.84     358.0       17.7   
        Case5    0.89     506.0       14.3    0.82     378.0       18.7   
        Case6    0.89     499.0       14.1    0.84     356.0       17.6   
        Case7    0.89     501.0       14.1    0.84     358.0       17.7   
        Case8    0.87     534.0       15.0    0.82     386.0       19.1   

Class             Wet                       Normal                       
              R2_Test RMSE_Test RRMSE_Test R2_Test RMSE_Test RRMSE_Test  
Group   Case                                                             
ASP+BAU Case1    0.76     516.0       11.9    0.77     575.0       12.6  
        Case2    0.77     500.0       11.6    0.78     568.0       12.4  
        Case3    0.76     508.0       11.8    0.79     562.0       12.3  
        Case4    0.76     508.0       11.8    0.78     564.0       12.3  
        Case5    0.77     500.0       11.6    0.78     570.0       12.4  
        Case6    0.77     501.0       11.6    0.78     562.0       12.3  
        Case7    0.77     501.0       11.6    0.79     559.0       12.2  
        Case8    0.71     561.0       13.0    0.74     614.0       13.4  
ASP     Case1    0.53     588.0       11.3    0.77     557.0       11.8  
        Case2    0.55     574.0       11.1    0.77     552.0       11.7  
        Case3    0.54     583.0       11.2    0.78     543.0       11.5  
        Case4    0.54     583.0       11.2    0.78     547.0       11.6  
        Case5    0.55     574.0       11.1    0.77     555.0       11.8  
        Case6    0.55     571.0       11.0    0.78     537.0       11.4  
        Case7    0.55     571.0       11.0    0.78     543.0       11.5  
        Case8    0.47     624.0       12.0    0.74     587.0       12.5  
BAU     Case1    0.72     395.0       10.5    0.79     582.0       13.1  
        Case2    0.71     399.0       10.6    0.80     572.0       12.9  
        Case3    0.72     391.0       10.4    0.80     563.0       12.7  
        Case4    0.72     391.0       10.4    0.80     563.0       12.7  
        Case5    0.71     399.0       10.6    0.80     572.0       12.9  
        Case6    0.72     389.0       10.4    0.80     567.0       12.7  
        Case7    0.72     389.0       10.4    0.80     567.0       12.7  
        Case8    0.63     449.0       12.0    0.77

Cross-validation scores (Supplementary Table S6).

In [5]:
table_s6 = results.pivot_table(index=["Group", "Case"], columns="Class", values=["R2_CV", "RMSE_CV", "RRMSE_CV"])
table_s6.to_csv(TABLE_DIR / "supp_table_s6_strategy1_cv.csv")
table_s6.xs("R2_CV", axis=1, level=0)[CLASSES].reindex(["ASP+BAU", "ASP", "BAU"], level=0).round(2)

Class           All   Dry   Wet  Normal
Group   Case                           
ASP+BAU Case1  0.91  0.85  0.78    0.77
        Case2  0.89  0.85  0.79    0.76
        Case3  0.91  0.85  0.79    0.77
        Case4  0.91  0.86  0.79    0.77
        Case5  0.91  0.85  0.79    0.76
        Case6  0.91  0.86  0.78    0.77
        Case7  0.91  0.86  0.78    0.77
        Case8  0.89  0.83  0.72    0.73
ASP     Case1  0.93  0.88  0.55    0.75
        Case2  0.91  0.87  0.54    0.75
        Case3  0.93  0.87  0.55    0.75
        Case4  0.93  0.88  0.55    0.75
        Case5  0.93  0.88  0.54    0.75
        Case6  0.93  0.88  0.54    0.75
        Case7  0.93  0.88  0.54    0.75
        Case8  0.92  0.84  0.40    0.72
BAU     Case1  0.88  0.82  0.72    0.76
        Case2  0.89  0.82  0.72    0.77
        Case3  0.89  0.84  0.71    0.77
        Case4  0.89  0.84  0.71    0.77
        Case5  0.89  0.83  0.72    0.77
        Case6  0.89  0.84  0.72    0.77
        Case7  0.90  0.84  0.72    0.77
        Case8  0.88  0.83  0.63    0.73

## Share of dry, normal and wet years in the train and test parts (Supplementary Table S5)

Same split as above (seed 42), applied to the row positions.

In [6]:
rows = []
for group, df in data.items():
    year_class = df["Year"].map(YEAR_CLASS)
    tr, te = train_test_split(np.arange(len(df)), test_size=0.25, random_state=42)
    for part, idx in [("Overall", np.arange(len(df))), ("Train", tr), ("Test", te)]:
        share = year_class.iloc[idx].value_counts(normalize=True) * 100
        rows.append({"Group": group, "Part": part, "N": len(idx),
                     **{f"{c} (%)": share.get(c, 0.0) for c in ["Dry", "Normal", "Wet"]}})
pd.DataFrame(rows).round(1)

,Group,Part,N,Dry (%),Normal (%),Wet (%)
0,ASP,Overall,19712,38.0,50.4,11.6
1,ASP,Train,14784,38.2,50.0,11.8
2,ASP,Test,4928,37.5,51.4,11.1
3,BAU,Overall,18985,31.6,49.9,18.4
4,BAU,Train,14238,31.8,49.8,18.4
5,BAU,Test,4747,31.2,50.3,18.4
6,ASP+BAU,Overall,38697,34.9,50.2,14.9
7,ASP+BAU,Train,29022,34.7,50.2,15.0
8,ASP+BAU,Test,9675,35.4,50.0,14.6


## Descriptive statistics (Supplementary Table S2) and correlation of the water variables

In [7]:
cols = [c for c in CASES["Case7"]] + [TARGET]
stats = []
for group in ["ASP", "BAU"]:
    d = data[group][cols].copy()
    d[TARGET] = d[TARGET] * Y_MULT
    s = d.agg(["mean", "std", "min", "max"]).T
    s.columns = [f"{group} {c}" for c in ["Mean", "SD", "Min", "Max"]]
    stats.append(s)
stats = pd.concat(stats, axis=1).rename(index={TARGET: "Yield (kg/ha)"})
stats.to_csv(TABLE_DIR / "supp_table_s2_descriptive_stats.csv")
stats.round(2)

,ASP Mean,ASP SD,ASP Min,ASP Max,BAU Mean,BAU SD,BAU Min,BAU Max
Carbon (0-15 cm),0.89,0.19,0.41,1.49,0.86,0.24,0.44,1.62
Carbon (15-30 cm),0.75,0.20,0.41,1.92,0.73,0.23,0.37,2.17
Clay (0-15 cm),24.34,2.76,17.07,30.66,24.40,3.66,14.63,30.46
Clay (15-30 cm),31.15,4.26,19.84,42.51,30.65,4.70,18.78,39.66
Clay (30-60 cm),29.68,3.39,18.56,42.80,29.57,4.23,17.05,44.54
Clay (60-90 cm),26.30,3.37,14.74,40.05,25.64,3.55,15.80,38.95
Clay (90-120 cm),23.33,2.98,14.86,35.58,23.12,3.41,13.39,37.93
Sand (0-15 cm),45.84,8.51,29.01,69.76,48.13,11.09,26.35,70.77
Sand (15-30 cm),41.85,7.74,27.77,65.01,42.92,10.16,27.46,68.72
Sand (30-60 cm),44.48,8.01,26.06,64.42,44.51,10.28,24.22,65.30


In [8]:
water = ["Precipitation", "ETa", "SM 30 cm", "SM 60 cm", "SM 90 cm"]
r = data["ASP+BAU"][water].corr(method="pearson")
print("Pearson r (square it for R2):")
r.round(2)

Pearson r (square it for R2):


,Precipitation,ETa,SM 30 cm,SM 60 cm,SM 90 cm
Precipitation,1.00,0.60,0.66,0.23,-0.03
ETa,0.60,1.00,0.38,0.24,0.16
SM 30 cm,0.66,0.38,1.00,-0.08,-0.16
SM 60 cm,0.23,0.24,-0.08,1.00,0.66
SM 90 cm,-0.03,0.16,-0.16,0.66,1.00
